# ueno bank: reproducible diagnostics

As of 12 November 2026. These are controlled sensitivities, not bank-calibrated forecasts. Read `review_es.md` for the corrections to the original draft.

The core model uses only the Python standard library. It writes CSVs and a JSON summary under `results/`. Outputs below were executed while building this notebook.

**Definitions:** normalized capital arithmetic; synthetic amortising cohorts with no 60-day classification lag; historical June loans/deposits sensitivity. No probabilities of failure or Monte Carlo calibration are claimed.

In [1]:
from pathlib import Path
__file__ = str(Path.cwd() / 'analysis.py')
"""Public-data diagnostics for a LinkedIn article, as of 2026-11-12.

Run: python analysis.py (Python 3.10+; stdlib only).
Not a reconstruction of Ueno's regulatory capital, loan tape or liquidity.
All synthetic parameters are explicit below. No estimated event probabilities.
"""
from pathlib import Path
import csv, itertools, json, math, unittest

ROOT = Path(__file__).resolve().parent
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)

def cohort_npl(growth=0.464, term=36, lifetime_pd=0.06,
               retention=12, scale=18.0, shape=2.0):
    """Exact expected stock for monthly equal-principal cohorts.

    Originations grow at (1+growth) annually indefinitely. Age zero is a new
    cohort at observation month. Defaults occur at months 1..term, with a
    Weibull timing distribution truncated/renormalised to term and multiplied
    by lifetime_pd. No competing prepayment. Performing principal amortises
    linearly; defaulted principal freezes at start-of-default-month EAD and
    stays in the gross book for retention monthly observations. Then exits
    (collection/writeoff/transfer unspecified), with no recovery model.
    These are expected stocks, not a random draw. No classification delay,
    acquired legacy pool, forbearance or bank-specific calibration is assumed.
    """
    if growth <= -1 or term < 1 or not 0 <= lifetime_pd <= 1 or retention < 1:
        raise ValueError('Invalid economic domain')
    cdf = lambda t: 1 - math.exp(-(t / scale) ** shape)
    masses = [0.] + [lifetime_pd * (cdf(d)-cdf(d-1))/cdf(term)
                      for d in range(1, term+1)]
    performing = npl = 0.
    for age in range(term + retention):
        origination = (1 + growth) ** (-age / 12)
        if age < term:
            survival = 1 - sum(masses[:age+1])
            performing += origination * (1 - age/term) * survival
        for d in range(max(1, age-retention+1), min(age, term)+1):
            ead = (term-d+1)/term
            npl += origination * masses[d] * ead
    return npl / (performing+npl)

def capital_ratio(c0=.15, correction=0., scenario_loss=0., rwa_change=0.):
    """Ratios and losses in opening RWA units; ignores tax/profit/actions.
    rwa_change is net total denominator change including provision effects.
    No bank-specific legal/economic capital conversion is implied.
    """
    if rwa_change <= -1: raise ValueError('Nonpositive RWA')
    return (c0-correction-scenario_loss)/(1+rwa_change)

def correction_budget(target=.12, c0=.15, scenario_loss=0., rwa_change=0.):
    return c0-scenario_loss-target*(1+rwa_change)

def funding_ratio(ldr=.919, runoff=0.):
    """Loans unchanged; customer deposits reduced; other funding ignored."""
    if not 0 <= runoff < 1: raise ValueError('Invalid runoff')
    return ldr/(1-runoff)

def write_csv(name, rows):
    with (OUT/name).open('w', newline='') as f:
        w=csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

class Checks(unittest.TestCase):
    def test_capital_identity_and_inversion(self):
        for g in [-.1,0,.1,.25]:
            d=correction_budget(rwa_change=g)
            self.assertAlmostEqual(capital_ratio(correction=d,rwa_change=g),.12)
        self.assertAlmostEqual(capital_ratio(),.15)
    def test_cohort_zero_defaults(self):
        self.assertEqual(cohort_npl(lifetime_pd=0),0)
    def test_cohort_invariants(self):
        for growth, term, window in itertools.product([0,.464,1],[24,36,60],[3,12,24]):
            value=cohort_npl(growth,term,.06,window)
            self.assertTrue(0 < value < 1)
            self.assertGreater(cohort_npl(growth,term,.10,window),value)
            self.assertGreaterEqual(cohort_npl(growth,term,.06,window+1),value)
    def test_retention_analytic_special_case(self):
        # One-month loans: performing age0 =1, default stock at age1 = PD.
        self.assertAlmostEqual(cohort_npl(0,1,.06,1),.06/1.06)
    def test_funding_identity(self):
        self.assertAlmostEqual(funding_ratio(runoff=1-.919),1)
        self.assertGreater(funding_ratio(runoff=.1),funding_ratio())

def main():
    suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
    result=unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful(): raise SystemExit(1)
    rows=[]
    for term,pd,window,scale,growth in itertools.product(
            [24,36,60],[.02,.06,.10],[3,6,12,24],[9.,18.,30.],[0.,.464,1.]):
        rows.append(dict(term_months=term,lifetime_pd=pd,retention_months=window,
                         weibull_scale=scale,growth=growth,
                         npl_ratio=cohort_npl(growth,term,pd,window,scale)))
    write_csv('cohort_sensitivity.csv',rows)
    core=[dict(term_months=t,growth=g,npl_ratio=cohort_npl(g,t))
          for t,g in itertools.product([36,60],[0,.464,1])]
    write_csv('cohort_core.csv',core)
    capital=[dict(rwa_change=g,target=t,
                  max_correction=correction_budget(target=t,rwa_change=g))
             for g,t in itertools.product([-.1,0,.1,.25],[.12,.06])]
    write_csv('capital_thresholds.csv',capital)
    funding=[dict(runoff=r,loan_deposit_ratio=funding_ratio(runoff=r))
             for r in [0,.05,.081,.10,.20,.30]]
    write_csv('funding_sensitivity.csv',funding)
    p=.06
    fitted=[]
    # Invert PD to match 0.4% for several unobserved exit/timing structures.
    for window,scale in itertools.product([3,6,12,24],[9.,18.,30.]):
        lo,hi=0.,1.
        for _ in range(70):
            mid=(lo+hi)/2
            if cohort_npl(lifetime_pd=mid,retention=window,scale=scale)<.004: lo=mid
            else: hi=mid
        fitted.append(dict(retention_months=window,weibull_scale=scale,
                           implied_lifetime_pd=(lo+hi)/2))
    write_csv('non_identification.csv',fitted)
    rel=[]
    for term,pd,window,scale in itertools.product([24,36,60],[.02,.06,.10],[3,6,12,24],[9.,18.,30.]):
        base=cohort_npl(0,term,pd,window,scale)
        grown=cohort_npl(.464,term,pd,window,scale)
        rel.append((grown/base-1)*100)
    summary=dict(as_of='2026-11-12',synthetic_sensitivity_rows=len(rows),
       cohort_core=core,relative_change_growth_46_4_pct=[min(rel),max(rel)],
       capital_thresholds=capital,funding_sensitivity=funding,
       pd_fitting_0_4_pct_range=[min(x['implied_lifetime_pd'] for x in fitted),max(x['implied_lifetime_pd'] for x in fitted)],
       fitch_ye25_capital_definition_gap_bps=round((.152-.103)*10000),
       reported_provisions_over_ye25_impaired_loans=1_621_853_458_972/(50_314*1e6),
       checks_run=result.testsRun,checks_passed=result.wasSuccessful())
    (OUT/'summary.json').write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary,indent=2))

if __name__=='__main__': main()


{
  "as_of": "2026-11-12",
  "synthetic_sensitivity_rows": 324,
  "cohort_core": [
    {
      "term_months": 36,
      "growth": 0,
      "npl_ratio": 0.02267590299447401
    },
    {
      "term_months": 36,
      "growth": 0.464,
      "npl_ratio": 0.01800845640020617
    },
    {
      "term_months": 36,
      "growth": 1,
      "npl_ratio": 0.01478787573483654
    },
    {
      "term_months": 60,
      "growth": 0,
      "npl_ratio": 0.01780177577223566
    },
    {
      "term_months": 60,
      "growth": 0.464,
      "npl_ratio": 0.016281248786229666
    },
    {
      "term_months": 60,
      "growth": 1,
      "npl_ratio": 0.0142099608402565
    }
  ],
  "relative_change_growth_46_4_pct": [
    -38.80090853605146,
    27.96616126919007
  ],
  "capital_thresholds": [
    {
      "rwa_change": -0.1,
      "target": 0.12,
      "max_correction": 0.041999999999999996
    },
    {
      "rwa_change": -0.1,
      "target": 0.06,
      "max_correction": 0.096
    },
    {
      "rwa

test_capital_identity_and_inversion (__main__.Checks.test_capital_identity_and_inversion) ... ok
test_cohort_invariants (__main__.Checks.test_cohort_invariants) ... ok
test_cohort_zero_defaults (__main__.Checks.test_cohort_zero_defaults) ... ok
test_funding_identity (__main__.Checks.test_funding_identity) ... ok
test_retention_analytic_special_case (__main__.Checks.test_retention_analytic_special_case) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.003s

OK


## Independent implementation check

A time-forward monthly ledger is compared with the direct age-stock calculation across all324 configurations. Reverse-PD solutions must match the target within1e-12.

In [2]:
"""Independent monthly ledger and inverse-solution validation."""
import math
from analysis import cohort_npl, ROOT, OUT

def ledger(growth,term,pd,window,scale):
    history=term+window+5
    cohorts=[]; defaults=[]
    normalizer=1-math.exp(-(term/scale)**2)
    for month in range(history):
        new=(1+growth)**((month-(history-1))/12)
        cohorts.append(dict(start=month,original=new,survivors=new))
        performing=0.
        for c in cohorts:
            age=month-c['start']
            if age>term: continue
            if age>0:
                prob=pd*(math.exp(-((age-1)/scale)**2)-math.exp(-(age/scale)**2))/normalizer
                default_units=c['original']*prob
                c['survivors']-=default_units
                defaults.append((month,default_units*(term-age+1)/term))
            performing+=c['survivors']*max(0,(term-age)/term)
        outstanding_default=sum(amount for t,amount in defaults if 0<=month-t<window)
    return outstanding_default/(performing+outstanding_default)

count=0; max_error=0
for g in [0,.464,1]:
 for term in [24,36,60]:
  for pd in [.02,.06,.1]:
   for window in [3,6,12,24]:
    for scale in [9,18,30]:
     expected=cohort_npl(g,term,pd,window,scale)
     independent=ledger(g,term,pd,window,scale)
     error=abs(expected-independent)
     assert error<1e-12,(g,term,pd,window,scale,error)
     max_error=max(max_error,error); count+=1
import csv
max_residual=0
with (OUT/'non_identification.csv').open() as f:
 for row in csv.DictReader(f):
  fit=cohort_npl(.464,36,float(row['implied_lifetime_pd']),int(row['retention_months']),float(row['weibull_scale']))
  max_residual=max(max_residual,abs(fit-.004))
assert max_residual<1e-12
text=(f'PASS: 5 model checks in analysis.py.\n'
      f'PASS: independent time-forward ledger matches all {count} cohort specifications.\n'
      f'Maximum absolute NPL ratio difference: {max_error:.3g}\n'
      f'PASS: 12 reverse-PD solutions have maximum residual {max_residual:.3g}.\n'
      'Domain and monotonicity checks passed. No bank calibration or empirical forecast validation is claimed.\n')
(ROOT/'VALIDATION.txt').write_text(text)
print(text)


PASS: 5 model checks in analysis.py.
PASS: independent time-forward ledger matches all 324 cohort specifications.
Maximum absolute NPL ratio difference: 5.13e-16
PASS: 12 reverse-PD solutions have maximum residual 1.73e-18.
Domain and monotonicity checks passed. No bank calibration or empirical forecast validation is claimed.



![Executed result](figures/cohort_growth.svg)

![Executed result](figures/capital_funding.svg)

## Interpretation limits

The reported46.4% is gross loan growth; mapping it to a stable origination regime is illustrative. Sensitivities do not reconstruct an acquired and rapidly changing loan book. The0.4% observed NPL reference is not directly calibrated to the synthetic default definition. Alternative reverse-PD fits are not confidence intervals. L/D above100% does not establish liquidity failure. Accounting equity, Fitch Core Capital and regulatory capital must be reconciled, not substituted.

References and dates are listed in`article.md`. Run`plot_results.py`with matplotlib to regenerate the supplied PNGs.